<a href="https://colab.research.google.com/github/fishmoonbird/WEL/blob/main/%E5%A8%81%E9%A9%AC1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
!pip -q install datasets transformers accelerate evaluate scikit-learn rank_bm25

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.5 MB/s eta 0:00:00


In [6]:
from datasets import load_dataset

ds = load_dataset("mteb/LCQMC")

print(ds)
print(ds["train"][0])

DatasetDict({
    train: Dataset({
        features: ['sentence1', 'sentence2', 'score'],
        num_rows: 238766
    })
    validation: Dataset({
        features: ['sentence1', 'sentence2', 'score'],
        num_rows: 8802
    })
    test: Dataset({
        features: ['sentence1', 'sentence2', 'score'],
        num_rows: 12500
    })
})
{'sentence1': '喜欢打篮球的男生喜欢什么样的女生', 'sentence2': '爱打篮球的男生喜欢什么样的女生', 'score': 1}


In [7]:
type(ds)

datasets.dataset_dict.DatasetDict

In [8]:
len(ds['train'])

238766

In [9]:
from transformers import AutoTokenizer
from transformers import AutoModelForSequenceClassification

model_name = "bert-base-chinese"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2
)

config.json:   0%|          | 0.00/624 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/110k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/269k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  412MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-chinese
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [10]:
def tokenize(batch):
    return tokenizer(
        batch["sentence1"],
        batch["sentence2"],
        truncation=True,
        max_length=128
    )

tokenized = ds.map(
    tokenize,
    batched=True
)

tokenized = tokenized.rename_column("score", "labels")

Map:   0%|          | 0/238766 [00:00<?, ? examples/s]

Map:   0%|          | 0/8802 [00:00<?, ? examples/s]

Map:   0%|          | 0/12500 [00:00<?, ? examples/s]

In [11]:
from datasets import load_dataset

ds = load_dataset("mteb/LCQMC")

print(ds)
print(ds["train"][0])

DatasetDict({
    train: Dataset({
        features: ['sentence1', 'sentence2', 'score'],
        num_rows: 238766
    })
    validation: Dataset({
        features: ['sentence1', 'sentence2', 'score'],
        num_rows: 8802
    })
    test: Dataset({
        features: ['sentence1', 'sentence2', 'score'],
        num_rows: 12500
    })
})
{'sentence1': '喜欢打篮球的男生喜欢什么样的女生', 'sentence2': '爱打篮球的男生喜欢什么样的女生', 'score': 1}


In [12]:
from transformers import TrainingArguments, Trainer, DataCollatorWithPadding

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

args = TrainingArguments(
    output_dir="./bert_faq",
    learning_rate=2e-5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    num_train_epochs=1,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    fp16=False
)

train_ds = tokenized["train"].shuffle(seed=42).select(range(30000))
dev_ds = tokenized["validation"].select(range(3000))

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=dev_ds,
    data_collator=data_collator,
)

trainer.train()

Epoch,Training Loss,Validation Loss
1,0.308521,0.411966


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=938, training_loss=0.2747965977390184, metrics={'train_runtime': 277.0703, 'train_samples_per_second': 108.276, 'train_steps_per_second': 3.385, 'total_flos': 809378939610240.0, 'train_loss': 0.2747965977390184, 'epoch': 1.0})

In [13]:
1 + 2

3

In [14]:
faq_list = list(
    set(ds["train"]["sentence2"])
)

faq_list = faq_list[:20000]

len(faq_list)

20000

In [15]:
from rank_bm25 import BM25Okapi

def chinese_tokenize(text):
    return list(text)

tokenized_faq = [
    chinese_tokenize(x)
    for x in faq_list
]

bm25 = BM25Okapi(tokenized_faq)

In [16]:
import numpy as np

def bm25_recall(query, topk=50):
    scores = bm25.get_scores(
        chinese_tokenize(query)
    )

    ids = np.argsort(scores)[::-1][:topk]

    return [
        (faq_list[i], scores[i])
        for i in ids
    ]

In [17]:
bm25_recall(
    "喜欢篮球的男生喜欢什么女生",
    10
)

[('男生喜欢女生有什么表现', np.float64(34.08528843273884)),
 ('为什么男生喜欢摸女生的乳', np.float64(33.52977384872894)),
 ('男生有腹肌，女生喜欢吗', np.float64(33.28075611837348)),
 ('女生喜欢男生有腹肌吗，', np.float64(33.28075611837348)),
 ('男生最喜欢女生送什么礼物', np.float64(32.84172882913039)),
 ('女生喜欢男生时会有什么表现？', np.float64(30.612159797944347)),
 ('男生为什么喜欢吸青女生的脖子？', np.float64(30.224183047682452)),
 ('初三男生喜欢什么生日礼物', np.float64(29.74033082977194)),
 ('问男生你们都喜欢女生什么发型？', np.float64(29.608762070515365)),
 ('女人喜欢男人大吗', np.float64(27.786862189999816))]

In [18]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import linear_kernel

vectorizer = TfidfVectorizer(
    analyzer="char",
    ngram_range=(2, 4),
    max_features=100000
)

faq_matrix = vectorizer.fit_transform(faq_list)

In [19]:
def tfidf_recall(query, topk=50):

    q = vectorizer.transform([query])

    score = linear_kernel(
        q,
        faq_matrix
    )[0]

    ids = score.argsort()[::-1][:topk]

    return [
        (faq_list[i], score[i])
        for i in ids
    ]

In [20]:
import torch

def bert_rerank(query, candidates):

    pairs = [
        (query, x)
        for x in candidates
    ]

    inputs = tokenizer(
        [x[0] for x in pairs],
        [x[1] for x in pairs],
        padding=True,
        truncation=True,
        max_length=128,
        return_tensors="pt"
    )

    inputs = {
        k: v.to(model.device)
        for k, v in inputs.items()
    }

    model.eval()

    with torch.no_grad():

        logits = model(**inputs).logits

        scores = torch.softmax(
            logits,
            dim=-1
        )[:, 1]

    result = list(
        zip(
            candidates,
            scores.cpu().numpy()
        )
    )

    return sorted(
        result,
        key=lambda x: x[1],
        reverse=True
    )

In [21]:
def faq_search(query):

    bm = bm25_recall(query, 50)
    tf = tfidf_recall(query, 50)

    candidates = list(
        dict.fromkeys(
            [x[0] for x in bm] +
            [x[0] for x in tf]
        )
    )

    ranking = bert_rerank(
        query,
        candidates
    )

    return ranking[:5]

In [22]:
faq_search("爱打篮球的男生喜欢什么样的女生")

[('王俊凯喜欢什么样的女生，讨厌什么样的女生', np.float32(0.020648727)),
 ('如何打好篮球，', np.float32(0.020378182)),
 ('女孩子一般最喜欢什么', np.float32(0.014147779)),
 ('男生喜欢女生有什么表现', np.float32(0.012855782)),
 ('什么样的女孩才算傻的可爱的', np.float32(0.010928073))]

In [23]:
faq_search("爱打篮球的人喜欢什么饮料")

[('如何打好篮球，', np.float32(0.013113587)),
 ('篮球是什么牌子的最好', np.float32(0.009457707)),
 ('我老婆喜欢什么啊', np.float32(0.007800174)),
 ('女孩子一般最喜欢什么', np.float32(0.00779498)),
 ('什么牌子篮球好', np.float32(0.0077003525))]